# Time series for newcomers: what generated series are for

Deep-RL papers often open with a tiny gridworld: far too simple to be useful, but it shows in one picture what the work is about. This notebook does the same for the time-series transformations of `kcai-data-sampling`, on the gridworld of time series: **Air Passengers**, twelve years of monthly airline passengers (Box & Jenkins, 1976), known from every forecasting textbook.

The story: an airline planner looks at the record and asks two questions.

1. **Forecasting**: how many passengers next year, month by month?
2. **Anomaly detection**: is this year normal, or did something happen?

The record is clean: twelve years, **no anomaly**. The transformations create the situations that are not in it. At the end, the **Nile** shows what a real anomaly looks like.

**Data.** Air Passengers and Nile, from the R `datasets` package through [Rdatasets](https://github.com/vincentarelbundock/Rdatasets), downloaded by `scripts/fetch_toy_samples.py`. Nile is public domain; the licence of Air Passengers is not established for certain (Rdatasets believes it free to redistribute), so the outputs of this notebook are kept out of the repository.

**Models.** The simplest forecasts there are, in `toy_models.py` next to this notebook. This is a toy, and it says so.

```bash
uv run --with matplotlib --with ipykernel jupyter lab
```

## Setup

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml

%config InlineBackend.figure_format = 'png'

# Work from the repository root: every path in the configs is relative to it.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
os.chdir(REPO)
sys.path.insert(0, str(REPO / "examples" / "notebooks"))

import toy_models as tm
from kcai_data_sampling_job.cli import run

DATA = Path("examples/data/toy_sample")
if not (DATA / "air_passengers.parquet").exists():
    subprocess.run([sys.executable, "scripts/fetch_toy_samples.py"], check=True)

FIGURES = Path("examples/outputs/toy-demo/figures")  # PNG copies for slides (git-ignored)
FIGURES.mkdir(parents=True, exist_ok=True)



def save(fig, name):
    fig.savefig(FIGURES / f"{name}.png", dpi=160, bbox_inches="tight")


LOG = tm.load_series(DATA / "air_passengers.parquet")  # log(thousands of passengers)
PASSENGERS = np.exp(LOG)
MONTHS = pd.date_range("1949-01", periods=len(LOG), freq="MS")

for name in ("toy-anomaly", "toy-forecasting"):
    with open(f"examples/config/{name}.yaml") as f:
        config = yaml.safe_load(f)
    shutil.rmtree(f"examples/outputs/{name}", ignore_errors=True)
    print(name, run(config))

## 1. A time series in one picture

144 months, and only two things to see: it **grows**, and it **repeats every year** (a summer peak).

The yearly swing grows with the traffic: 44 thousand passengers in 1949, 232 thousand in 1960. So we describe changes **in percent** (technically, we work on the logarithm of the series): "10 % fewer passengers" means the same thing in 1949 and in 1960, "30 thousand fewer" does not. On the right, the same series in percent: the yearly swing keeps its size.

In [ ]:
yearly = PASSENGERS.reshape(-1, 12).mean(axis=1)
mid_year = pd.date_range("1949-07", periods=12, freq="12MS")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
ax = axes[0]
ax.plot(MONTHS, PASSENGERS, label="monthly passengers")
ax.plot(mid_year, yearly, marker="o", ms=4, label="yearly mean: it grows")
ax.annotate("summer peak, every year", (MONTHS[127], PASSENGERS[127]), xytext=(MONTHS[70], 560), fontsize=8, arrowprops={"arrowstyle": "-", "color": "gray", "lw": 0.8})
ax.set_title("Air Passengers, 1949-1960", loc="left")
ax.set_ylabel("thousands of passengers")
ax.legend(loc="upper left")
ax = axes[1]
ax.plot(MONTHS, 100 * (LOG - LOG[0]))
ax.set_title("The same series in percent (log scale)", loc="left")
ax.set_ylabel("change since January 1949 (log %)")
fig.tight_layout()
save(fig, "1_series")
plt.show()

## 2. A transformation is a situation

The series is cut into **windows** of 36 months, one starting every month. A transformation takes a window and returns a modified copy; each one stands for something that could happen to the airline:

| Transformation | Situation | Parameter |
|---|---|---|
| `jitter` | counting noise | `sigma` 0.02: about 2 % noise |
| `level_shift` | demand drops at once and stays down (a lasting strike, a new competitor) | `offset` -0.19: about -17 %, from month 27 of the window |
| `drift` | demand slowly collapses | `slope` -0.02: about -2 % per month |
| `add_sinusoid` (one cycle per year) | the season is reshaped | `amplitude` 0.2: about ±20 % |

Here is one window, 1955-1957, recorded (blue) and generated (orange).

In [ ]:
ANOMALY, ANOMALY_WINDOWS = tm.load_outputs("examples/outputs/toy-anomaly/air_passengers")
START = 72  # the window starting in January 1955
SHOWN = [
    ("jitter", 0.02, "Counting noise (jitter, about 2 %)"),
    ("level_shift", -0.19, "Demand drops by 17 % (level_shift)"),
    ("drift", -0.02, "Demand slowly collapses (drift, -2 %/month)"),
    ("add_sinusoid", 0.2, "Season reshaped (add_sinusoid, ±20 %)"),
]
in_window = ANOMALY.start_window == START
months = MONTHS[START : START + 36]

fig, axes = plt.subplots(2, 2, figsize=(11, 5.2), sharex=True, sharey=True)
for ax, (algorithm, value, title) in zip(axes.flat, SHOWN):
    row = ANOMALY.index[in_window & (ANOMALY.algorithm == algorithm) & np.isclose(ANOMALY.value, value)][0]
    ax.plot(months, PASSENGERS[START : START + 36], label="recorded")
    ax.plot(months, np.exp(ANOMALY_WINDOWS[row]), label="generated")
    ax.set_title(title, loc="left", fontsize=10)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
for ax in axes[:, 0]:
    ax.set_ylabel("thousands")
axes[0, 0].legend(loc="upper left")
fig.tight_layout()
save(fig, "2_situations")
plt.show()

Every generated window keeps a record of where it comes from and what was done to it, in the **ledger**: the source window (`start_window`, `end_window`), the algorithm, its parameters and the seed. Here are the four windows above:

In [ ]:
rows = [ANOMALY.index[in_window & (ANOMALY.algorithm == a) & np.isclose(ANOMALY.value, v)][0] for a, v, _ in SHOWN]
ANOMALY.loc[rows, ["series_id", "start_window", "end_window", "algorithm", "params", "seed"]]

## 3. Forecasting next year

Two forecasts a newcomer can do by hand:

- **seasonal naive**: next year is this year;
- **seasonal naive with growth**: next year is this year, plus the growth between the last two years.

They need two years of history. We degrade the history (counting noise, smoothed reporting) and forecast the next twelve months; the **true next year is read from the recorded series**, so it is never transformed. Because we work in percent, the error reads as a percentage.

In [ ]:
FORECAST, FORECAST_WINDOWS = tm.load_outputs("examples/outputs/toy-forecasting/air_passengers")
keep = (FORECAST.end_window + 12 < len(LOG)).to_numpy()
FORECAST, FORECAST_WINDOWS = FORECAST[keep].reset_index(drop=True), FORECAST_WINDOWS[keep]
TRUTH = np.stack([LOG[e + 1 : e + 13] for e in FORECAST.end_window])
MODELS = {"seasonal naive with growth": tm.seasonal_growth, "seasonal naive": tm.seasonal_naive}
for name, model in MODELS.items():
    FORECAST[name] = 100 * np.abs(model(FORECAST_WINDOWS) - TRUTH).mean(axis=1)
ERRORS = FORECAST.groupby(["algorithm", "value"])[list(MODELS)].mean().round(1)
ERRORS

In [ ]:
END = 107  # history up to December 1957, forecast of 1958
clean = FORECAST.index[(FORECAST.end_window == END) & (FORECAST.algorithm == "jitter") & np.isclose(FORECAST.value, 0.0)][0]
noisy = FORECAST.index[(FORECAST.end_window == END) & (FORECAST.algorithm == "jitter") & np.isclose(FORECAST.value, 0.1)][0]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
ax = axes[0]
history, future = MONTHS[END - 23 : END + 1], MONTHS[END + 1 : END + 13]
ax.plot(history, PASSENGERS[END - 23 : END + 1], label="recorded history")
ax.plot(future, PASSENGERS[END + 1 : END + 13], lw=2.2, label="true 1958")
ax.plot(future, np.exp(tm.seasonal_growth(FORECAST_WINDOWS[[clean]])[0]), ls="--", label="forecast, clean history")
ax.plot(future, np.exp(tm.seasonal_growth(FORECAST_WINDOWS[[noisy]])[0]), ls="--", label="forecast, 10 % noise in the history")
ax.set_title("Forecasting 1958 from 1956-1957", loc="left")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.set_ylabel("thousands")
ax.legend(loc="upper left")
ax = axes[1]
curve = ERRORS.loc["jitter"]
for model in MODELS:
    ax.plot(curve.index * 100, curve[model], marker="o", ms=5, label=model)
ax.set_title("Forecast error vs. counting noise", loc="left")
ax.set_xlabel("noise in the history (about %)")
ax.set_ylabel("mean error on the next year (%)")
ax.set_ylim(bottom=0)
ax.legend(loc="lower right")
fig.tight_layout()
save(fig, "3_forecasting")
plt.show()

Adding the growth halves the error of the plain seasonal forecast, and the noisier the history, the worse both get: the generated histories turn "how robust is my forecast?" into a curve.

## 4. No anomaly in the record? We make them

A toy detector: forecast the last year of each 36-month window from the two years before (seasonal naive with growth), and raise an alert when the error is larger than on 95 % of the recorded windows. So about 5 % of normal windows raise an alert, by construction.

The record has no labelled anomaly. The label comes from the ledger: we know which fault was injected, of what size, into which window.

In [ ]:
DETECTOR = tm.Detector().fit(tm.cut(LOG, 36))
ANOMALY["alert"] = DETECTOR.alerts(ANOMALY_WINDOWS)
FALSE_ALARMS = DETECTOR.alerts(tm.cut(LOG, 36)).mean()
RATES = ANOMALY.groupby(["algorithm", "value"]).alert.mean()

curve = RATES.loc["level_shift"].sort_index(ascending=False) * 100  # smallest drop first
drops = 100 * (1 - np.exp(curve.index.to_numpy()))
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.axhline(FALSE_ALARMS * 100, color="gray", lw=1, ls="--")
ax.text(drops.max(), FALSE_ALARMS * 100 + 2, "false alarms on recorded windows ", ha="right", color="gray", fontsize=8)
ax.plot(drops, curve.values, marker="o", ms=6)
for x, y in zip(drops, curve.values):
    ax.annotate(f"{y:.0f} %", (x, y), textcoords="offset points", xytext=(0, 8), ha="center", fontsize=8)
ax.set_title("Demand drop (level_shift): windows raising an alert", loc="left")
ax.set_xlabel("size of the drop (%)")
ax.set_ylabel("windows raising an alert (%)")
ax.set_ylim(0, 112)
fig.tight_layout()
save(fig, "4_detection")
plt.show()

pd.DataFrame({
    "fault": ["demand drop -17 % (level_shift)", "slow collapse -2 %/month (drift)", "season reshaped ±20 % (add_sinusoid)", "counting noise 2 % (jitter, normal)"],
    "windows raising an alert": [RATES.loc[("level_shift", -0.19)], RATES.loc["drift"].max(), RATES.loc["add_sinusoid"].max(), RATES.loc[("jitter", 0.02)]],
}).style.format({"windows raising an alert": "{:.0%}"}).hide()

- A **sudden drop** is caught once it is large enough: we read off the smallest drop the detector sees.
- A **slow collapse** and a **reshaped season** are not caught at all, whatever their size: their alert rate stays exactly at the false-alarm level. The detector copies last year's shape, so a reshaped season is simply forecast; and it estimates the growth from the history, so a steady collapse is absorbed as "the new trend". These faults are invisible to it by construction, and the generated faults reveal this blind spot before a real event does.

## 5. What a real anomaly looks like

The Nile at Aswan, 1871-1970: around 1899 its yearly flow drops at once and stays lower (the first Aswan dam was built in those years). That is a **level shift**, the kind of break `level_shift` generates.

In [ ]:
NILE = tm.load_series(DATA / "nile.parquet", column="values")
years = np.arange(1871, 1871 + len(NILE))
row = ANOMALY.index[in_window & (ANOMALY.algorithm == "level_shift") & np.isclose(ANOMALY.value, -0.19)][0]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
ax = axes[0]
ax.plot(years, NILE)
for a, b in ((1871, 1898), (1899, 1970)):
    ax.hlines(NILE[(years >= a) & (years <= b)].mean(), a, b, color="C1", lw=2.2)
ax.set_title(f"A real break: the Nile, mean {NILE[years <= 1898].mean():,.0f} → {NILE[years >= 1899].mean():,.0f}", loc="left", fontsize=10)
ax.set_ylabel("yearly flow (10⁸ m³)")
ax = axes[1]
ax.plot(months, PASSENGERS[START : START + 36], label="recorded")
ax.plot(months, np.exp(ANOMALY_WINDOWS[row]), label="generated: -17 % from month 27")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.set_title("A generated break: Air Passengers", loc="left", fontsize=10)
ax.set_ylabel("thousands")
ax.legend(loc="upper left")
fig.tight_layout()
save(fig, "5_nile")
plt.show()

## 6. What this shows

- **A transformation is a situation**, described with a few physical parameters ("17 % fewer passengers from month 27").
- **Generated data is labelled data**: the ledger says what was done to each window, so a record with no anomaly becomes a test set with known anomalies.
- **Sweeping a parameter gives a curve**: how fast a forecast degrades, the smallest fault a detector sees, and what it cannot see at all.

The same transformations, configs and ledger run on real data in `ett-demo.ipynb` (oil temperature of power transformers): the gridworld first, then the real environment.